# DDL 操作
## 数据库相关操作
## 查看数据库

In [12]:

from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

### 列出所有数据库

In [13]:
existed_databases = client.list_databases()

for db in existed_databases:
      print(db)

rag_demo
default


### 创建数据库

In [14]:
db_name = "rag_demo"
if db_name not in existed_databases:
      client.create_database(db_name=db_name)

### 删除数据库
如果数据库下有collection则无法删除  需要先删除所有的collection才能删除Database

In [ ]:
client.drop_database(db_name=db_name)

## Collection 相关的操作

In [7]:
client.use_database(db_name=db_name)

### 查看数据库下联的collections

In [15]:
collection = client.list_collections()
for coll in collection:
      print(coll)

### 创建collection

In [16]:
collection_name = "docs"
client.create_collection(
      collection_name=collection_name,
      dimension=2560,
      metric_type="COSINE",

)

### 删除collection

In [11]:
client.drop_collection(collection_name=collection_name)

## DML操作
### 嵌入模型的初始化

In [17]:
import os

from dotenv import load_dotenv
from langchain_ollama import OllamaEmbeddings

load_dotenv(override=True)

# 初始化嵌入模型
embedding_model = OllamaEmbeddings(
      model="qwen3-embedding:4b",
      base_url=os.getenv("OLAMA_BASE_URL"),
)


### 准备collection

In [18]:
collection_name = "docs"
client.create_collection(
      collection_name=collection_name,
      dimension=2560,
      metric_type="COSINE",
)

### 查看collection 元数据

In [19]:
from rich import print as rprint

metadata = client.describe_collection(collection_name=collection_name)
rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 2560}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 470607125433787107,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC', 'namespace.sharding.enabled': 'false', 'max_field_id': '102'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 470642956010258450,
    'update_timestamp': 470642956010258450
}

### 准备数据

In [24]:
texts = [
      "LangChain是一个用于构建LLM 应用的开发框架。",
      "Milvus是一个适合 AI 应用的向量数据库。",
      "RAG的核心是先检索相关知识，再让大模型生成答案",
      "Docker Desktop 可以方便地在本地运行 Milvus Standalone。",
]

### 生成嵌入向量

In [25]:
vectors = embedding_model.embed_documents(texts)

### 查看生成的嵌入向量

In [26]:
print(len(vectors))

print(len(vectors[0]))

print((vectors[0][:5]))

4
2560
[-0.00024173092, -0.048800994, 0.00286426, 0.03596787, -0.00088123576]


### 封装为可以插入的数据格式

In [27]:
data = [
      {
            "id": i,
            "vector": vectors[i],
            "text": texts[i],
            "source": "demo"
      } for i in range(len(vectors))
]

### 写入数据

In [28]:
insert_res = client.upsert(
      collection_name=collection_name,
      data=data,
)
print("insert_res:", insert_res)

insert_res: {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


### 手动flush
Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush，将数据刷写到磁盘

In [29]:
client.flush(collection_name=collection_name)

查看collection统计的信息

In [30]:
stata = client.get_collection_stats(collection_name=collection_name)
print("stata:", stata)

stata: {'row_count': 5}


## DQL操作
扫描数据

In [33]:
iterator = client.query_iterator(
      collection_name=collection_name,
      filter="",
      output_fields=["*"]
)

i = 0
while True:
      rows = iterator.next()

      if not rows:
            break
      for row in rows:
            print(f"第{i + 1}条数据")
            # print(row)
            i += 1
            print(f"id : {row["id"]},vector = {row["vector"][:5]},text ={row["text"]}, source = {row["source"]} ")
            iterator.close()

第1条数据
id : 0,vector = [-0.00024173091514967382, -0.0488009937107563, 0.0028642599936574697, 0.0359678715467453, -0.0008812357555143535],text =LangChain是一个用于构建LLM 应用的开发框架。, source = demo 
第2条数据
id : 1,vector = [-0.00023824212257750332, -0.01790534518659115, -0.010629955679178238, 0.03622710704803467, -0.0009254419128410518],text =Milvus是一个适合 AI 应用的向量数据库。, source = demo 
第3条数据
id : 2,vector = [-0.00040596030885353684, -0.021360499784350395, -0.007841771468520164, 0.023270197212696075, -0.0019151559099555016],text =RAG的核心是先检索相关知识，再让大模型生成答案, source = demo 
第4条数据
id : 3,vector = [-0.00042794691398739815, -0.0052576689049601555, -0.05388420447707176, 0.032272033393383026, -0.001973411999642849],text =Docker Desktop 可以方便地在本地运行 Milvus Standalone。, source = demo 


### 通过主键来查询

## 相似度检索

In [34]:
query = "什么是向量数据库"
query_vector = embedding_model.embed_query(query)

### 检索

In [38]:
results = client.search(
      collection_name=collection_name,
      data=[query_vector],
      limit=3,
      output_fields=["text"]
)
for res in results[0]:
      print(res)

{'id': 1, 'distance': 0.7091843485832214, 'entity': {'text': 'Milvus是一个适合 AI 应用的向量数据库。'}}
{'id': 2, 'distance': 0.5385993719100952, 'entity': {'text': 'RAG的核心是先检索相关知识，再让大模型生成答案'}}
{'id': 3, 'distance': 0.46878552436828613, 'entity': {'text': 'Docker Desktop 可以方便地在本地运行 Milvus Standalone。'}}
